# Exploring Coral Bleaching Risk Using NOAA Coral Reef Watch DHW Data

**Author:** Frank Velez  
**Date:** May 2026  
**Kernel:** Python (coral-dhw)  
**Data:** NOAA Coral Reef Watch Version 3.1 Daily 5km Satellite Coral Bleaching Heat Stress Product Suite  
**Source:** https://coralreefwatch.noaa.gov/  
**Citation:** NOAA CRW (2000, updated daily). CoralTemp and Coral Bleaching Heat Stress. Accessed May 2026.

## Project Motivation & Dataset Rationale

When I started looking for a dataset to anchor this first marine science project,
I had a simple question: *what data actually gets used by the people trying to
protect coral reefs in the real world?* That question led me directly to NOAA's
Coral Reef Watch (CRW) program and their Degree Heating Weeks (DHW) product — and
then to the paper that describes exactly how it's built: Skirving et al. (2020),
*CoralTemp and the Coral Reef Watch Coral Bleaching Heat Stress Product Suite
Version 3.1.*

Reading that paper changed how I thought about this project.

### Why not just use raw sea surface temperature?

My first instinct was to pull raw SST data — it's widely available from multiple
sources including NOAA OISST, NASA MODIS, and the Copernicus Marine Service. But
raw SST alone doesn't tell you whether a coral is stressed. A reading of 29°C
means something very different on the Great Barrier Reef versus Bonaire. What
matters isn't the absolute temperature — it's how that temperature compares to
what the corals at that location have adapted to over their lifetimes.

That realization pushed me toward NOAA CRW's derived product suite, where that
comparison is already built in — and rigorously validated against decades of field
bleaching observations worldwide.

### The climatological baseline — and why it matters so much

The entire CRW product suite rests on a climatological baseline: a set of
long-term average SST values for every location on Earth, for every month of the
year. This is not a trivial thing to build correctly. As Skirving et al. (2020)
describe, CRW spent years grappling with a fundamental problem — the satellite SST
data used to *build* the climatology came from different sensors and processing
pipelines than the near-real-time data used to *calculate* the anomalies. That
mismatch introduced systematic biases, particularly at the high temperatures most
relevant to bleaching.

The development of **CoralTemp** — a single, continuous daily SST record spanning
1985 to the present, built by carefully blending three different satellite products
— was specifically designed to solve this problem. For the first time, CRW could
compute the climatology and the daily anomalies from the *same* underlying dataset.
The Monthly Mean (MM) climatology was derived from this record across 1985–2012,
and the **Maximum Monthly Mean (MMM)** — the warmest month at each location —
became the critical threshold from which the HotSpot and DHW products are derived.

I want to be transparent about why this matters for this project: I am not
independently validating the baseline. I am trusting that NOAA CRW's 25-year
investment in getting this right is sound — which is supported by the peer-reviewed
literature and by the fact that these products are the global standard used by reef
managers worldwide. But understanding *that the baseline exists, how it was
constructed, and what its assumptions are* is essential context for interpreting
everything downstream. The DHW values I analyze are only as meaningful as the MMM
they're derived from.

### The 5km pixel limitation

One more thing worth acknowledging upfront: the Virtual Station data used in this
analysis represents a single 5×5 km satellite pixel per reef location. Real reef
systems are far more heterogeneous than that. Temperatures vary with depth, local
current patterns, reef orientation, and bathymetry — a coral colony in a deep
channel experiences very different conditions than one on a shallow exposed crest,
even within the same reef system.

This means the DHW values I analyze are best understood as a **reef-wide surface
thermal stress index** — a strong and validated predictor of bleaching risk at the
population level, but not a precise measurement of conditions at any specific
location. This is the same limitation the entire global reef monitoring community
works with, and it doesn't undermine the analysis. It just means I should be
careful about over-interpreting site-level differences that might reflect local
thermal refugia or exposure patterns rather than true differences in bleaching
pressure.

With those caveats clearly on the table, here is what this project actually does.

## Setup & Configuration

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import os
from io import StringIO

# ── Directory setup ──────────────────────────────────────────
CACHE_DIR = "data/raw"
FIG_DIR   = "figures"
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(FIG_DIR,   exist_ok=True)

# ── Bleaching thresholds (Skirving et al. 2020, Liu et al. 2013) ──
DHW_WARNING  = 4   # Alert Level 1: Bleaching Likely
DHW_SEVERE   = 8   # Alert Level 2: Mortality Likely

print("✓ Libraries loaded")
print(f"✓ Cache directory: {CACHE_DIR}")
print(f"✓ Figures directory: {FIG_DIR}")

✓ Libraries loaded
✓ Cache directory: data/raw
✓ Figures directory: figures


## Study Sites

## Study Sites & Station ID Verification

Five reef systems were selected to compare thermal stress patterns across two ocean 
basins. Site selection prioritized locations with well-documented bleaching histories, 
geographic diversity within the Caribbean, and one Indo-Pacific site for cross-basin 
comparison.

### How station IDs were identified

NOAA CRW Virtual Station data are accessed via a simple URL pattern:
    https://coralreefwatch.noaa.gov/product/vs/data/{station_id}.txt
Station IDs were **verified directly from the live NOAA CRW Virtual Station list** at:
[https://coralreefwatch.noaa.gov/product/vs/data.php](https://coralreefwatch.noaa.gov/product/vs/data.php)

This step was intentional — station slugs were not assumed or inferred from site 
names, but confirmed against the authoritative NOAA source before any data was 
fetched. Four corrections were made from initial estimates:

| Site | Initial Assumption | Verified Station ID | Note |
|------|--------------------|---------------------|------|
| Florida Keys | `florida_keys` | `florida_keys` | ✓ Confirmed |
| Grand Cayman | `cayman_islands_grand_cayman` | `cayman_islands` | Covers all three Cayman Islands |
| Roatán, Honduras | `mesoamerican_reef_roatan` | `honduras` | No Roatán-specific station; Honduras station covers the Bay Islands |
| Bonaire | `bonaire` | `abc_islands` | Bonaire grouped with Aruba and Curaçao |
| Great Barrier Reef | `great_barrier_reef` | `gbr_central` | GBR has no single station — split into 5 sectors; central chosen as most studied |

This verification step is important for reproducibility — anyone running this
notebook can confirm station IDs independently at the URL above.   

One finding during verification deserves special mention. Our initial assumption 
was that the Great Barrier Reef would have a single station — `great_barrier_reef` 
— mirroring the pattern of the other sites. When the fetch returned a 404, I went 
back to the NOAA CRW station page and discovered that the GBR is simply too large 
to be represented by a single regional pixel. NOAA splits it into five sectors: 
Torres Strait, Far Northern, Northern, Central, and Southern. This is actually a 
meaningful scientific distinction — thermal stress patterns must vary considerably across 
the reef's 2,300 km length. For this analysis I selected **GBR Central** 
(`gbr_central`), which covers the Whitsundays and Townsville region. 

This sector has been extensively studied. I'll be honest — my awareness of the 
2016 GBR bleaching event came not from a journal paper but from watching Jeff 
Orlowski's documentary *Chasing Coral*, which follows researchers documenting 
that exact event in real time. That film is what pointed me toward this region 
as the most relevant GBR sector to include. I later found that the 2016 event 
was also the subject of Hughes et al. (2017) in *Nature* — so the documentary 
and the peer-reviewed literature were pointing at the same place.

## Data Acquisition — Fetch Function

### A note on the file format
Before writing the data fetch function, we (me and AI) took the time to inspect the raw file 
directly from NOAA's servers rather than assuming a format and hoping for the best. 
This turned out to be an important step.

The NOAA CRW Virtual Station files are not simple CSVs. Each file opens with an 
unstructured metadata header block containing station-level information including 
the station name, coordinates, and critically — the **Averaged Maximum Monthly Mean 
(MMM)** temperature. This is the climatological baseline I discussed earlier: the 
warmest month that corals at this location are adapted to. It is embedded directly 
in the file header, which means I can extract it programmatically and carry it 
through the analysis as a reference value.

The actual data table begins at line 21 (for Florida Keys), headed by a single 
column header row: YYYY MM DD SST_MIN SST_MAX SST@90th_HS SSTA@90th_HS 90th_HS>0 DHW_from_90th_HS>1 BAA_7day_max

A few things worth noting as someone new to this data:

1. **The date is split across three columns** — `YYYY`, `MM`, `DD` — not a single 
   date string. I combine them into a proper `date` column during parsing.

2. **The column names reflect the updated v3.1 product suite.** DHW is labeled 
   `DHW_from_90th_HS>1` — meaning it accumulates HotSpot values exceeding 1°C above 
   the 90th percentile SST, consistent with the methodology described in 
   Skirving et al. (2020). The bleaching alert level is `BAA_7day_max` — the 
   7-day maximum Bleaching Alert Area value.

3. **The header block uses plain text labels, not `#` comment markers.** My first 
   attempt at parsing failed because I filtered lines starting with `#`, which is 
   a common convention — but NOAA's format doesn't follow it. Inspecting the raw 
   file first saved me from silently loading malformed data.

The fetch function below handles all of this: it parses the header to extract the 
MMM, finds the data start line dynamically, builds a proper date column, and renames 
the columns to friendlier labels for analysis.

In [11]:
SITES = {
    "Florida Keys": {
        "station_id": "florida_keys",
        "lat": 24.55,
        "lon": -81.50,
        "ocean": "Caribbean/Atlantic",
    },
    "Cayman Islands": {
        "station_id": "cayman_islands",
        "lat": 19.30,
        "lon": -81.38,
        "ocean": "Caribbean",
    },
    "Honduras (Bay Islands)": {
        "station_id": "honduras",
        "lat": 16.32,
        "lon": -86.55,
        "ocean": "Caribbean",
    },
    "ABC Islands (Bonaire)": {
        "station_id": "abc_islands",
        "lat": 12.15,
        "lon": -68.28,
        "ocean": "Caribbean",
    },
    "GBR Central": {
        "station_id": "gbr_central",
        "lat": -18.00,
        "lon": 147.00,
        "ocean": "Indo-Pacific",
    },
}

for name, cfg in SITES.items():
    print(f"✓ {name:30} | station: {cfg['station_id']}")

✓ Florida Keys                   | station: florida_keys
✓ Cayman Islands                 | station: cayman_islands
✓ Honduras (Bay Islands)         | station: honduras
✓ ABC Islands (Bonaire)          | station: abc_islands
✓ GBR Central                    | station: gbr_central


## Data Acquisition

Data are fetched live from the NOAA CRW Virtual Station feed using the confirmed 
station IDs above. The fetch function:

- Builds the URL from the station ID
- Downloads the raw text file
- Skips comment lines (lines starting with `#`)
- Parses the remaining data into a pandas DataFrame
- Caches the result locally to `data/raw/` so the notebook works offline after first run

To force a fresh download, set `force_refresh=True` in the fetch call.

> **Note:** The cell below was our first attempt at parsing the CRW data file. 
> It failed because we (me and I) assumed the file used `#` comment markers and a single 
> date column — neither of which turned out to be true. It is preserved here 
> as a record of the discovery process. See the corrected implementation below.

In [8]:
# Inspect the raw file before parsing
with open("data/raw/florida_keys.txt", "r") as f:
    lines = f.readlines()

print(f"Total lines: {len(lines)}")
print("\n--- First 20 lines ---")
for i, line in enumerate(lines[:20]):
    print(f"{i:3}: {repr(line)}")

Total lines: 15127

--- First 20 lines ---
  0: 'Name:\n'
  1: 'Florida Keys\n'
  2: ' \n'
  3: 'Polygon Middle Longitude:\n'
  4: '-81.6250 \n'
  5: ' \n'
  6: 'Polygon Middle Latitude:\n'
  7: '24.7500 \n'
  8: ' \n'
  9: 'Averaged Maximum Monthly Mean:\n'
 10: '29.6264\n'
 11: ' \n'
 12: 'Averaged Monthly Mean (Jan-Dec):\n'
 13: '23.4166 23.1889 23.5492 24.6089 26.4747 28.3325 29.3254 29.6264 29.1674 27.7384 26.0846 24.4925\n'
 14: ' \n'
 15: 'First Valid DHW Date:\n'
 16: '1985 25 03\n'
 17: ' \n'
 18: 'First Valid BAA Date:\n'
 19: '1985 31 03\n'


In [9]:
print("--- Lines 20-40 ---")
for i, line in enumerate(lines[20:40], start=20):
    print(f"{i:3}: {repr(line)}")

--- Lines 20-40 ---
 20: ' \n'
 21: 'YYYY MM DD SST_MIN SST_MAX SST@90th_HS SSTA@90th_HS 90th_HS>0 DHW_from_90th_HS>1 BAA_7day_max\n'
 22: '1985 01 01 24.4100 25.1500 24.9600      0.5113       0.0000    0.0000            0\n'
 23: '1985 01 02 24.1200 25.1300 24.9500      0.4029       0.0000    0.0000            0\n'
 24: '1985 01 03 23.9900 25.2100 24.9800      0.4281       0.0000    0.0000            0\n'
 25: '1985 01 04 23.0500 25.1500 24.6300      0.2032       0.0000    0.0000            0\n'
 26: '1985 01 05 22.6200 24.9400 24.3500     -0.1681       0.0000    0.0000            0\n'
 27: '1985 01 06 22.3600 25.2600 24.7700      0.2471       0.0000    0.0000            0\n'
 28: '1985 01 07 21.9500 24.9600 24.4100      0.1474       0.0000    0.0000            0\n'
 29: '1985 01 08 21.8700 24.9100 24.2300     -0.1061       0.0000    0.0000            0\n'
 30: '1985 01 09 21.6900 24.6100 23.9500     -0.3619       0.0000    0.0000            0\n'
 31: '1985 01 10 21.7600 24.4700 23.79

In [12]:
CRW_BASE_URL = "https://coralreefwatch.noaa.gov/product/vs/data/{station_id}.txt"

def parse_crw_header(lines):
    """Extract metadata from the CRW file header block."""
    meta = {}
    for i, line in enumerate(lines):
        line = line.strip()
        if line == "Averaged Maximum Monthly Mean:":
            meta["mmm"] = float(lines[i+1].strip())
        if line == "Name:":
            meta["name"] = lines[i+1].strip()
        if line.startswith("YYYY MM DD"):
            meta["data_start_line"] = i
            break
    return meta

def fetch_crw_station(site_name, station_id, force_refresh=False):
    cache_path = os.path.join(CACHE_DIR, f"{station_id}.txt")

    if not force_refresh and os.path.exists(cache_path):
        print(f"  ✓ Loaded from cache: {site_name}")
        with open(cache_path, "r") as f:
            raw_lines = f.readlines()
    else:
        url = CRW_BASE_URL.format(station_id=station_id)
        print(f"  ↓ Fetching live: {site_name}")
        print(f"    {url}")
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        raw_lines = resp.text.splitlines(keepends=True)
        with open(cache_path, "w") as f:
            f.writelines(raw_lines)

    # Parse header metadata
    meta = parse_crw_header(raw_lines)
    data_start = meta["data_start_line"]

    # Parse data table — starts at header line, data follows
    data_text = "".join(raw_lines[data_start:])
    df = pd.read_csv(
        StringIO(data_text),
        sep=r"\s+",
        na_values=["-999", "-9999", "-99.0000"],
    )

    # Build a proper date column from the three separate YYYY MM DD columns
    df["date"] = pd.to_datetime(
        df["YYYY"].astype(str) + "-" +
        df["MM"].astype(str).str.zfill(2) + "-" +
        df["DD"].astype(str).str.zfill(2)
    )

    # Rename key columns to friendly names
    df.rename(columns={
        "SST@90th_HS":         "sst",
        "SSTA@90th_HS":        "sst_anomaly",
        "90th_HS>0":           "hotspot",
        "DHW_from_90th_HS>1":  "dhw",
        "BAA_7day_max":        "alert",
    }, inplace=True)

    df["site"] = site_name
    df["mmm"]  = meta.get("mmm", None)

    # Keep only the columns I need
    keep = ["date", "sst", "sst_anomaly", "hotspot", "dhw", "alert", "site", "mmm"]
    df = df[[c for c in keep if c in df.columns]]
    df.sort_values("date", inplace=True)
    df.reset_index(drop=True, inplace=True)

    print(f"    ✓ {len(df):,} rows | "
          f"{df['date'].min().date()} → {df['date'].max().date()} | "
          f"MMM: {meta.get('mmm')}°C")
    return df


# ── Fetch all sites ───────────────────────────────────────────
print("Fetching NOAA CRW Virtual Station data...\n")
all_data = {}
for site_name, cfg in SITES.items():
    df = fetch_crw_station(site_name, cfg["station_id"])
    all_data[site_name] = df

print(f"\n✓ Done. {len(all_data)} sites loaded.")

Fetching NOAA CRW Virtual Station data...

  ✓ Loaded from cache: Florida Keys
    ✓ 15,105 rows | 1985-01-01 → 2026-05-10 | MMM: 29.6264°C
  ✓ Loaded from cache: Cayman Islands
    ✓ 15,105 rows | 1985-01-01 → 2026-05-10 | MMM: 29.3925°C
  ✓ Loaded from cache: Honduras (Bay Islands)
    ✓ 15,105 rows | 1985-01-01 → 2026-05-10 | MMM: 28.87°C
  ✓ Loaded from cache: ABC Islands (Bonaire)
    ✓ 15,105 rows | 1985-01-01 → 2026-05-10 | MMM: 28.0193°C
  ↓ Fetching live: GBR Central
    https://coralreefwatch.noaa.gov/product/vs/data/gbr_central.txt
    ✓ 15,105 rows | 1985-01-01 → 2026-05-10 | MMM: 28.3422°C

✓ Done. 5 sites loaded.


## First Look at the Data

All five sites loaded successfully — 15,105 daily records per site, spanning 
1985-01-01 through May 2026. The live fetch confirms our station IDs are correct 
and the data pipeline is working end to end.

### The MMM values are already telling a story

Before plotting a single chart, the Averaged Maximum Monthly Mean temperatures 
extracted from the file headers are worth pausing on:

| Site | MMM (°C) |
|------|----------|
| Florida Keys | 29.63 |
| Cayman Islands | 29.39 |
| Honduras (Bay Islands) | 28.87 |
| GBR Central | 28.34 |
| ABC Islands (Bonaire) | 28.02 |

This is the MMM doing exactly what Skirving et al. (2020) described — capturing 
location-specific thermal tolerance rather than absolute temperature. The Florida 
Keys corals are adapted to a warmer baseline than GBR Central corals. This means 
that a raw SST reading of 29°C would represent no stress at all in the Florida 
Keys, but would already be pushing past the bleaching threshold on the GBR. 

This is precisely why I chose the CRW derived products over raw SST — and why 
the MMM baseline is the foundation everything else is built on.